In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)

In [4]:
#load engineered datasets
X_train = pd.read_csv("X_train_engineered.csv")
X_test = pd.read_csv("X_test_engineered.csv")

y_train = pd.read_csv("y_train.csv").squeeze()
y_test = pd.read_csv("y_test.csv").squeeze()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (36168, 22)
X_test shape: (9043, 22)
y_train shape: (36168,)
y_test shape: (9043,)


In [5]:
#inspect data types
X_train.dtypes

age                            int64
job                              str
marital                          str
education                        str
default                          str
balance                        int64
housing                          str
loan                             str
contact                          str
day                            int64
month                            str
duration                       int64
campaign                       int64
pdays                          int64
previous                       int64
poutcome                         str
age_group                        str
has_balance                    int64
balance_to_age               float64
has_previous_contact           int64
previous_positive_outcome      int64
contact_intensity                str
dtype: object

In [6]:
#Identify numerical and categorical columns automatically
numerical_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous', 'has_balance', 'balance_to_age', 'has_previous_contact', 'previous_positive_outcome']

Categorical columns:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'age_group', 'contact_intensity']


C:\Users\kone\AppData\Local\Temp\ipykernel_1276\1509486484.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X_train.select_dtypes(


In [7]:
#Check missing values
missing_values = X_train.isnull().sum()

missing_values = missing_values[
    missing_values > 0
].sort_values(ascending=False)

missing_values

Series([], dtype: int64)

In [8]:
#Build preprocessing pipelines
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

In [9]:
#Combine steps
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numeric_pipeline,
            numerical_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ]
)

In [10]:
#Fit on training data
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (36168, 64)
Processed testing shape: (9043, 64)


In [11]:
#Get the processed feature names
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

feature_names

Number of processed features: 64


array(['numerical__age', 'numerical__balance', 'numerical__day',
       'numerical__duration', 'numerical__campaign', 'numerical__pdays',
       'numerical__previous', 'numerical__has_balance',
       'numerical__balance_to_age', 'numerical__has_previous_contact',
       'numerical__previous_positive_outcome', 'categorical__job_admin.',
       'categorical__job_blue-collar', 'categorical__job_entrepreneur',
       'categorical__job_housemaid', 'categorical__job_management',
       'categorical__job_retired', 'categorical__job_self-employed',
       'categorical__job_services', 'categorical__job_student',
       'categorical__job_technician', 'categorical__job_unemployed',
       'categorical__job_unknown', 'categorical__marital_divorced',
       'categorical__marital_married', 'categorical__marital_single',
       'categorical__education_primary',
       'categorical__education_secondary',
       'categorical__education_tertiary',
       'categorical__education_unknown', 'categorical__

In [12]:
#convert back to dataframes
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print(X_train_processed_df.shape)
print(X_test_processed_df.shape)

X_train_processed_df.head()

(36168, 64)
(9043, 64)


,numerical__age,numerical__balance,numerical__day,numerical__duration,numerical__campaign,numerical__pdays,numerical__previous,numerical__has_balance,numerical__balance_to_age,numerical__has_previous_contact,numerical__previous_positive_outcome,categorical__job_admin.,categorical__job_blue-collar,categorical__job_entrepreneur,categorical__job_housemaid,categorical__job_management,categorical__job_retired,categorical__job_self-employed,categorical__job_services,categorical__job_student,categorical__job_technician,categorical__job_unemployed,categorical__job_unknown,categorical__marital_divorced,categorical__marital_married,categorical__marital_single,categorical__education_primary,categorical__education_secondary,categorical__education_tertiary,categorical__education_unknown,categorical__default_no,categorical__default_yes,categorical__housing_no,categorical__housing_yes,categorical__loan_no,categorical__loan_yes,categorical__contact_cellular,categorical__contact_telephone,categorical__contact_unknown,categorical__month_apr,categorical__month_aug,categorical__month_dec,categorical__month_feb,categorical__month_jan,categorical__month_jul,categorical__month_jun,categorical__month_mar,categorical__month_may,categorical__month_nov,categorical__month_oct,categorical__month_sep,categorical__poutcome_failure,categorical__poutcome_other,categorical__poutcome_success,categorical__poutcome_unknown,categorical__age_group_18-25,categorical__age_group_26-35,categorical__age_group_36-45,categorical__age_group_46-55,categorical__age_group_56-65,categorical__age_group_66+,categorical__contact_intensity_High,categorical__contact_intensity_Low,categorical__contact_intensity_Medium
0,-0.460434,-0.164410,1.582124,-0.457311,-0.246104,-0.410910,-0.241509,0.437473,-0.135761,-0.471755,-0.185648,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-1.589641,0.899627,-1.298384,2.502493,0.398202,1.446096,2.664584,0.437473,1.833332,2.119745,-0.185648,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.292371,-0.365486,-0.458236,5.697691,0.398202,-0.410910,-0.241509,0.437473,-0.383660,-0.471755,-0.185648,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.668773,-0.445003,1.822166,-0.862499,2.653271,-0.410910,-0.241509,-2.285854,-0.458971,-0.471755,-0.185648,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,-0.272233,-0.361249,1.222060,-0.777602,2.331118,-0.410910,-0.241509,0.437473,-0.367443,-0.471755,-0.185648,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [13]:
#vwrify there are no missing values
print(
    "Missing values in training data:",
    X_train_processed_df.isnull().sum().sum()
)

print(
    "Missing values in testing data:",
    X_test_processed_df.isnull().sum().sum()
)

Missing values in training data: 0
Missing values in testing data: 0


In [14]:
X_train_processed_df.to_csv(
    "X_train_processed.csv",
    index=False
)

X_test_processed_df.to_csv(
    "X_test_processed.csv",
    index=False
)

y_train.to_csv(
    "y_train_processed.csv",
    index=False
)

y_test.to_csv(
    "y_test_processed.csv",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.
